# Breachpoint v2: Granite analyst batch on vLLM (Google Colab)

Runs **IBM Granite 4.2-3B** behind **vLLM's OpenAI-compatible server** over ~137k vulnerability descriptions and saves structured fields (product category, internet-facing, login needed, attack type...).

**Before you start**
1. *Runtime → Change runtime type → **T4 GPU***
2. Copy `granite_job.zip` (from `outputs/v2_Granite_Analyst/` in the repo) to `MyDrive/breachpoint_granite/` on Google Drive. It holds `queue.parquet`, `prompt.py`, `extract.py`.
3. Results go to Google Drive (`MyDrive/breachpoint_granite/`), so a disconnect loses at most one chunk. **Re-run all cells to resume**; finished rows are skipped.

Expected: a few minutes to start, then roughly 1.5-3 hours for the full queue on a T4. Rows run in priority order, so a partial run still covers the most likely vulnerabilities in every cohort.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/breachpoint_granite'
zipfile.ZipFile(f'{WORK}/granite_job.zip').extractall(WORK)
%cd {WORK}
!ls -la
!grep -n "PROMPT_VERSION =" prompt.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Upload granite_job.zip


In [ ]:
!pip install -q "vllm>=0.20" openai pandas pyarrow

## Start the vLLM server
T4 GPUs don't support bfloat16, so the model runs in float16 (`--dtype half`). Prefix caching reuses the long, shared system prompt across requests.

In [ ]:
!nohup vllm serve ibm-granite/granite-4.2-3b --served-model-name granite-4.2-3b \
    --dtype half --max-model-len 4096 --gpu-memory-utilization 0.92 \
    --enable-prefix-caching --max-num-seqs 128 > /content/vllm.log 2>&1 &

In [ ]:
import time, urllib.request
for i in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2); print('vLLM ready'); break
    except Exception:
        time.sleep(10)
else:
    print('Not ready after 20 min; see the log below')
!tail -n 20 /content/vllm.log

## Pilot (60 rows): check the prompt before the full run
20 known later-exploited vulns + 40 random ones from 2024-25. Check that `ok` is ~100% (valid JSON) and the fields look right.
`pilot_results.csv` is saved to the Drive folder for checking by hand.
If outputs look garbled in float16, restart the server with `--dtype float32` (slower).

In [ ]:
!python extract.py --queue pilot_queue.parquet --out pilot_run --concurrency 32 --chunk 60
import pandas as pd, glob
t = pd.concat(pd.read_parquet(p) for p in glob.glob('pilot_run/part-*.parquet'))
t = t.merge(pd.read_parquet('pilot_queue.parquet')[['cve_id', 'description']], on='cve_id')
print('valid JSON:', t.ok.mean())
t.to_csv('pilot_results.csv', index=False)
print('Saved to Drive:', WORK + '/pilot_results.csv')
pd.set_option('display.max_colwidth', 120)
t[['cve_id', 'category', 'internet_facing', 'pre_auth', 'attack_type', 'server_product', 'product', 'description']].head(20)

## Full run (resumable)

In [ ]:
!python extract.py --queue queue.parquet --out granite_fields --concurrency 128 --chunk 2000

## Package the results
Get `granite_fields.zip` from the Drive folder and unzip it into `outputs/v2_Granite_Analyst/` in the repo (or copy the folder from Drive).

In [ ]:
!cd {WORK} && zip -qr granite_fields.zip granite_fields && ls -la granite_fields.zip
import pandas as pd, glob
r = pd.concat(pd.read_parquet(p) for p in glob.glob(f'{WORK}/granite_fields/part-*.parquet'))
print(len(r), 'rows,', round(r.ok.mean()*100, 1), '% valid'); print(r.category.value_counts().head(10))
print('Saved to Drive:', WORK + '/granite_fields.zip')